# 03c — Batch-hard mining + adaptive per-writer thresholds (EfficientNet)

Builds directly on **03b** (the current best model: fine-tuned EfficientNet-B0, invert
preprocessing, writer-independent split, combined Latin+Devanagari set). 3c adds the two
FAR-targeting techniques chosen from the literature review:

1. **Online batch-hard triplet mining (training).** Drop 3b's pre-built offline triplets.
   Each batch samples P writers × K images (genuine **and** their forgeries); for every genuine
   anchor we mine the hardest positive and hardest negative *inside the batch*. Because a
   writer's own forgeries are in the batch, the hardest negative is usually a **skilled
   forgery** — so training focuses exactly where false-accepts come from.
2. **Adaptive per-writer threshold (inference).** Replace the single global EER threshold with a
   per-writer threshold derived from each writer's **enrollment references only** (their natural
   genuine spread). This targets the NFI FAR≈40% problem, which is a *calibration* failure, not
   a ranking one. The α multiplier is tuned on **validation** and applied unchanged to test.

**Carried over from 3b unchanged (the parts that helped):** fine-tuned EfficientNet-B0 tower,
SigNet invert preprocessing, no-flip augmentation, writer-independent split, combined dataset,
per-script threshold (kept as a baseline), NFI cross-dataset eval, per-epoch Drive checkpoint.

> Honest note: batch-hard mining is genuinely more code than 3b (a PK sampler + a custom
> `train_step`). That complexity is intrinsic to the technique, not decoration — everything
> outside the mining loop is kept as simple as 3b.

In [ ]:
import os, json, math, random, csv
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import Model
from keras.layers import Input, Dense, Dropout, Lambda
from keras.applications import EfficientNetB0
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve

## Data — same combined set and writer-independent split as 3b / NB3

In [ ]:
!git clone https://github.com/goyashek/Signature-forgery-verification.git

DATA_ROOT = 'Signature-forgery-verification/sign_data_combined'
if not os.path.isdir(DATA_ROOT):                      # running locally, not on Colab
    DATA_ROOT = 'sign_data_combined'
MANIFEST = os.path.join(DATA_ROOT, 'manifest.csv')

IMG_H, IMG_W = 155, 220
print('data root:', DATA_ROOT)

In [ ]:
rows = list(csv.DictReader(open(MANIFEST)))

genuine, forg = defaultdict(list), defaultdict(list)
script_of = {}
for r in rows:
    (genuine if r['kind'] == 'genuine' else forg)[r['writer']].append(r['relpath'])
    script_of[r['writer']] = r['script']

writers = sorted(set(genuine) & set(forg))
icdar = [w for w in writers if w.startswith('icdar_')]
bhh   = [w for w in writers if w.startswith('bhh_')]
num   = lambda w: int(w.split('_')[1])

train_w = [w for w in icdar if num(w) <= 40]        + [w for w in bhh if num(w) <= 110]
val_w   = [w for w in icdar if 41 <= num(w) <= 48]  + [w for w in bhh if 111 <= num(w) <= 130]
test_w  = [w for w in icdar if num(w) >= 49]        + [w for w in bhh if num(w) >= 131]

assert not (set(train_w) & set(val_w)), 'train/val writer leak'
assert not (set(train_w) & set(test_w)), 'train/test writer leak'
assert not (set(val_w) & set(test_w)), 'val/test writer leak'

def _c(ws): return f"{len(ws)} (icdar {sum(w.startswith('icdar_') for w in ws)} + bhh {sum(w.startswith('bhh_') for w in ws)})"
print('train:', _c(train_w)); print('val:  ', _c(val_w)); print('test: ', _c(test_w))

## Eval pairs (for the baseline / per-script comparison)

Training uses batch-hard mining (below), but evaluation still uses the leak-free 3-recipe
pairs from NB3/3b so the global and per-script thresholds stay directly comparable across
notebooks.

In [ ]:
def make_pairs(wset, per_writer, seed=1):
    rng = random.Random(seed); wl = sorted(wset); out = []
    for w in wl:
        g = genuine[w]
        if len(g) < 2: continue
        scr = script_of[w]
        for _ in range(per_writer):                                # match (0)
            a, b = rng.sample(g, 2); out.append((a, b, 0, scr))
        for _ in range(per_writer // 2):                           # forgery negative (1)
            if forg.get(w): out.append((rng.choice(g), rng.choice(forg[w]), 1, scr))
        for _ in range(per_writer // 2):                           # different-writer negative (1)
            other = rng.choice([x for x in wl if x != w])
            out.append((rng.choice(g), rng.choice(genuine[other]), 1, scr))
    rng.shuffle(out)
    return pd.DataFrame(out, columns=['img1', 'img2', 'label', 'script'])

pairs_val  = make_pairs(val_w,  per_writer=40, seed=2)
pairs_test = make_pairs(test_w, per_writer=40, seed=3)
print('val pairs:', len(pairs_val), '| test pairs:', len(pairs_test))
print('test by script:', pairs_test['script'].value_counts().to_dict())

## Image pipeline — identical to 3b (invert + 3-channel, no-flip augmentation)

In [ ]:
_CACHE = {}                                            # relpath -> decoded uint8 grayscale (bg=255)

def load_gray(relpath):
    im = _CACHE.get(relpath)
    if im is None:
        im = cv2.imread(os.path.join(DATA_ROOT, relpath), cv2.IMREAD_GRAYSCALE)
        im = cv2.resize(im, (IMG_W, IMG_H)); _CACHE[relpath] = im
    return im

def augment(im, rng):
    ang  = rng.uniform(-5, 5)
    tx   = rng.uniform(-0.06, 0.06) * IMG_W
    ty   = rng.uniform(-0.06, 0.06) * IMG_H
    zoom = rng.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((IMG_W / 2, IMG_H / 2), ang, zoom)
    M[0, 2] += tx; M[1, 2] += ty
    return cv2.warpAffine(im, M, (IMG_W, IMG_H), borderValue=255)   # white fill = background

def to3(im):
    """uint8 HxW grayscale -> inverted float32 HxWx3 (SigNet invert; EfficientNet scales internally)."""
    inv = 255.0 - im.astype('float32')
    return np.repeat(inv[..., None], 3, axis=2)

class PairSequence(keras.utils.Sequence):
    """Eval only: yields (X1, X2); never augments. shuffle=False keeps predict aligned."""
    def __init__(self, frame, batch_size=64, root=DATA_ROOT):
        try:    super().__init__(workers=4, use_multiprocessing=False, max_queue_size=12)
        except TypeError: super().__init__()
        self.f = frame.reset_index(drop=True); self.bs = batch_size; self.root = root
    def __len__(self): return math.ceil(len(self.f) / self.bs)
    def _prep(self, relpath):
        im = cv2.imread(os.path.join(self.root, relpath), cv2.IMREAD_GRAYSCALE)
        return to3(cv2.resize(im, (IMG_W, IMG_H)))
    def __getitem__(self, i):
        sub = self.f.iloc[i*self.bs:(i+1)*self.bs]
        return (np.stack([self._prep(p) for p in sub['img1']]),
                np.stack([self._prep(p) for p in sub['img2']]))

## PK sampler for batch-hard mining

Each batch holds **P writers × K images**. For each writer we draw `K_GEN` genuine + `K_FORG`
of their own forgeries; remaining slots up to K are filled with more genuine. Labels carried
alongside each image are `(writer_index, is_genuine)` — the custom `train_step` uses them to
build the positive/negative masks. Genuine images of a writer are the only valid **anchors**;
forgeries appear only as negatives (a skilled forgery of A is a hard negative for A).

In [ ]:
P_WRITERS = 8          # writers per batch
K_GEN     = 4          # genuine per writer  (anchors + positives)
K_FORG    = 2          # own forgeries per writer (hard negatives)
K_PER     = K_GEN + K_FORG
BATCH     = P_WRITERS * K_PER

w2idx = {w: i for i, w in enumerate(sorted(train_w))}

class PKSequence(keras.utils.Sequence):
    """Yields (X, y) where y[:,0]=writer index, y[:,1]=is_genuine. Augments (train)."""
    def __init__(self, wset, steps_per_epoch=120, seed=0):
        try:    super().__init__(workers=4, use_multiprocessing=False, max_queue_size=12)
        except TypeError: super().__init__()
        self.wl = [w for w in sorted(wset) if len(genuine[w]) >= K_GEN]
        self.steps = steps_per_epoch; self.rng = random.Random(seed)
    def __len__(self): return self.steps
    def _prep(self, relpath):
        return to3(augment(load_gray(relpath), self.rng))
    def __getitem__(self, i):
        X, yw, yg = [], [], []
        chosen = self.rng.sample(self.wl, min(P_WRITERS, len(self.wl)))
        for w in chosen:
            g = self.rng.sample(genuine[w], K_GEN)
            f = (self.rng.sample(forg[w], min(K_FORG, len(forg[w]))) if forg.get(w) else [])
            f += self.rng.choices(genuine[w], k=K_PER - K_GEN - len(f))   # pad with genuine
            for p in g: X.append(self._prep(p)); yw.append(w2idx[w]); yg.append(1.0)
            for p in f:
                is_g = 1.0 if p in genuine[w] else 0.0
                X.append(self._prep(p)); yw.append(w2idx[w]); yg.append(is_g)
        y = np.stack([np.array(yw, 'float32'), np.array(yg, 'float32')], axis=1)
        return np.stack(X).astype('float32'), y

train_gen = PKSequence(train_w, steps_per_epoch=120)
print('batch size:', BATCH, '| steps/epoch:', len(train_gen))

## Tower — fine-tuned EfficientNet-B0 (identical to 3b)

In [ ]:
def l2(t): return tf.math.l2_normalize(t, axis=1)

def build_tower():
    base = EfficientNetB0(include_top=False, weights='imagenet',
                          input_shape=(IMG_H, IMG_W, 3), pooling='avg')
    base.trainable = True
    inp = Input(shape=(IMG_H, IMG_W, 3))
    x = base(inp); x = Dropout(0.3)(x); x = Dense(128)(x)
    x = Lambda(l2, name='l2')(x)
    return Model(inp, x, name='tower')

tower = build_tower()
print('tower params:', f"{tower.count_params():,}")

## Batch-hard triplet loss + a custom training model

For embeddings `E` in a batch, compute all pairwise squared Euclidean distances, then per
genuine anchor *i*:
- **hardest positive** = farthest *j* with same writer and both genuine,
- **hardest negative** = closest *j* that is **not** "same writer & both genuine" (so a
  skilled forgery of the same writer counts as a negative),
- loss_i = `relu(d_hardpos − d_hardneg + margin)`, averaged over valid anchors.

This is the Hermans et al. batch-hard loss, adapted so a writer's own forgeries are negatives.

In [ ]:
MARGIN = 0.3

def batch_hard_loss(emb, y):
    w = y[:, 0]; g = y[:, 1]                                  # writer idx, is_genuine
    r = tf.reduce_sum(tf.square(emb), axis=1)
    D2 = tf.maximum(r[:, None] - 2.0 * tf.matmul(emb, emb, transpose_b=True) + r[None, :], 0.0)

    same_w = tf.equal(w[:, None], w[None, :])
    gi = g[:, None] > 0.5; gj = g[None, :] > 0.5
    both_gen = tf.logical_and(gi, gj)
    eye = tf.eye(tf.shape(emb)[0], dtype=tf.bool)

    pos_mask = tf.logical_and(tf.logical_and(same_w, both_gen), tf.logical_not(eye))
    neg_mask = tf.logical_and(tf.logical_not(tf.logical_and(same_w, both_gen)),
                              tf.logical_not(eye))

    pos_d = tf.where(pos_mask, D2, tf.zeros_like(D2))
    hard_pos = tf.reduce_max(pos_d, axis=1)                   # farthest positive
    BIG = tf.reduce_max(D2) + 1.0
    neg_d = tf.where(neg_mask, D2, BIG * tf.ones_like(D2))
    hard_neg = tf.reduce_min(neg_d, axis=1)                   # closest negative

    loss_row = tf.maximum(hard_pos - hard_neg + MARGIN, 0.0)
    valid = tf.logical_and(g > 0.5, tf.reduce_any(pos_mask, axis=1))   # genuine anchor w/ a positive
    valid = tf.cast(valid, tf.float32)
    return tf.reduce_sum(loss_row * valid) / (tf.reduce_sum(valid) + 1e-8)

class BatchHardModel(keras.Model):
    def __init__(self, tower):
        super().__init__()
        self.tower = tower
        self.loss_tracker = keras.metrics.Mean(name='loss')
    @property
    def metrics(self): return [self.loss_tracker]
    def call(self, x, training=False): return self.tower(x, training=training)
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            emb = self.tower(x, training=True)
            loss = batch_hard_loss(emb, tf.cast(y, tf.float32))
        grads = tape.gradient(loss, self.tower.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.tower.trainable_weights))
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}

model = BatchHardModel(tower)
model.compile(optimizer=Adam(1e-4))

## Checkpoint to Drive, then train (Colab free-tier safe)

Weights are saved to Drive every epoch; re-running the cells reloads and resumes after a
disconnect. Set `INITIAL_EPOCH` to the last completed epoch when resuming.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/sigver_nb3c'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT = os.path.join(CKPT_DIR, 'batchhard_3c.weights.h5')

# build once so weights exist before any load
_ = tower(np.zeros((1, IMG_H, IMG_W, 3), 'float32'))
if os.path.exists(CKPT):
    tower.load_weights(CKPT); print('resumed weights from', CKPT)
else:
    print('no checkpoint yet — training from ImageNet init')

In [ ]:
INITIAL_EPOCH = 0        # set to last completed epoch when resuming
EPOCHS = 30

checkpoint = ModelCheckpoint(CKPT, save_weights_only=True, save_freq='epoch', verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, verbose=1)
history = model.fit(train_gen, epochs=EPOCHS, initial_epoch=INITIAL_EPOCH,
                    callbacks=[checkpoint, reduce_lr])

In [ ]:
plt.plot(history.history['loss']); plt.title('Batch-hard triplet loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()

## Baseline eval — global + per-script thresholds (same protocol as 3b)

In [ ]:
def pair_distances(frame, root=DATA_ROOT):
    gen = PairSequence(frame, batch_size=64, root=root); e1, e2 = [], []
    for i in range(len(gen)):
        X1, X2 = gen[i]
        e1.append(tower.predict(X1, verbose=0)); e2.append(tower.predict(X2, verbose=0))
    e1 = np.concatenate(e1); e2 = np.concatenate(e2)
    return np.sqrt(np.sum((e1 - e2) ** 2, axis=1) + 1e-12)

def eer_threshold(d, y):
    fpr, tpr, thr = roc_curve(y, d); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr)); return float(thr[i]), float((fpr[i] + fnr[i]) / 2)

val_d = pair_distances(pairs_val);  y_va = pairs_val['label'].to_numpy(); scr_va = pairs_val['script'].to_numpy()
test_d = pair_distances(pairs_test); y_te = pairs_test['label'].to_numpy(); scr_te = pairs_test['script'].to_numpy()

threshold, val_eer = eer_threshold(val_d, y_va)
print('global threshold:', round(threshold, 4), '| val EER', round(val_eer*100, 2), '% | val AUC', round(roc_auc_score(y_va, val_d), 3))

def report_t(name, d, y, t):
    if len(y) == 0 or len(np.unique(y)) < 2: print(f'{name:12s} — n/a'); return
    pred = (d > t).astype(int)
    print(f'{name:12s} | n={len(y):5d} | AUC {roc_auc_score(y, d):.3f} | acc {100*(pred==y).mean():5.2f}% | '
          f'FAR {100*(d[y==1] < t).mean():5.2f}% | FRR {100*(d[y==0] > t).mean():5.2f}%')

print('\nglobal threshold on test:'); report_t('overall', test_d, y_te, threshold)
per_script_thr = {s: eer_threshold(val_d[scr_va==s], y_va[scr_va==s])[0] for s in ['latin','devanagari']}
print('\nper-script thresholds on test:')
for s in ['latin','devanagari']:
    m = scr_te == s; report_t(s.capitalize(), test_d[m], y_te[m], per_script_thr[s])

## Adaptive per-writer threshold (the new inference contribution)

Verification protocol per writer: hold out `N_REF` genuine signatures as **enrollment
references**, score every query by its **mean distance to those references**, and accept if the
score is below a writer-specific threshold

    τ_w = mean_intra + α · std_intra

where `mean_intra`/`std_intra` are computed **only from the enrollment references' pairwise
distances** (the writer's natural genuine spread) — never from queries or forgeries, so the
threshold is fair. Queries are the writer's remaining genuine (label 0) and their forgeries
(label 1). The single knob **α is tuned on the validation writers** (minimising EER) and applied
unchanged to test — no test-set peeking.

In [ ]:
def writer_eval(wset, n_ref=5, alpha=1.0, seed=11):
    """Per-writer enrollment protocol. Returns (scores, labels, per-writer decisions at alpha)."""
    rng = random.Random(seed); scores, labels = [], []
    accepts_genuine = rejects_forgery = n_gen = n_forg = 0
    for w in sorted(wset):
        g = genuine[w]
        if len(g) < n_ref + 1: continue
        refs = rng.sample(g, n_ref)
        ref_emb = tower.predict(np.stack([to3(load_gray(p)) for p in refs]), verbose=0)
        # intra spread from refs only -> threshold
        rd = np.sqrt(np.maximum(np.sum((ref_emb[:,None]-ref_emb[None,:])**2, -1), 0))
        iu = np.triu_indices(n_ref, 1); intra = rd[iu]
        tau = intra.mean() + alpha * intra.std()
        def score(relpath):
            e = tower.predict(to3(load_gray(relpath))[None], verbose=0)[0]
            return float(np.sqrt(np.sum((e - ref_emb)**2, axis=1) + 1e-12).mean())
        for p in [x for x in g if x not in refs]:           # genuine queries (label 0)
            s = score(p); scores.append(s); labels.append(0)
            n_gen += 1; accepts_genuine += (s < tau)
        for p in forg.get(w, []):                            # forgery queries (label 1)
            s = score(p); scores.append(s); labels.append(1)
            n_forg += 1; rejects_forgery += (s >= tau)
    scores = np.array(scores); labels = np.array(labels)
    far = 100*(1 - rejects_forgery/max(n_forg,1)); frr = 100*(1 - accepts_genuine/max(n_gen,1))
    return scores, labels, far, frr

# tune alpha on validation (minimise |FAR-FRR|), then apply to test
print('tuning alpha on validation writers...')
best = None
for a in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    _, _, far, frr = writer_eval(val_w, alpha=a)
    gap = abs(far - frr)
    print(f'  alpha={a:.1f}  FAR {far:5.2f}%  FRR {frr:5.2f}%  |gap| {gap:5.2f}')
    if best is None or gap < best[1]: best = (a, gap)
alpha = best[0]
print('chosen alpha (from val):', alpha)

In [ ]:
# apply the val-chosen alpha to the held-out TEST writers
s_te, l_te, far_te, frr_te = writer_eval(test_w, alpha=alpha)
auc_pw = roc_auc_score(l_te, s_te)
print('--- per-writer adaptive threshold on TEST (alpha from val) ---')
print(f'AUC {auc_pw:.3f} | FAR {far_te:.2f}% | FRR {frr_te:.2f}%  (n={len(l_te)} queries)')

## Save tower + meta (for the Streamlit app)

In [ ]:
tower.save('siamese_bh_embedding.keras')
json.dump({'global_threshold': threshold, 'per_writer_alpha': alpha, 'n_ref': 5,
           'img_h': IMG_H, 'img_w': IMG_W, 'preprocess': 'invert_gray3ch', 'channels': 3,
           'model': 'siamese_efficientnet_b0_batchhard'},
          open('siamese_bh_meta.json', 'w'))
print('saved tower + meta')

## NFI cross-dataset test — pair-based AND per-writer

The pair-based numbers are directly comparable to NB2 / NB3 / 3b. The per-writer protocol is
where the adaptive threshold should help most, since NFI's global-threshold FAR was ≈40%.

In [ ]:
import glob, re

NFI_DIR = None
for c in ['sign_data_nfi', '../sign_data_nfi', 'Signature-forgery-verification/sign_data_nfi']:
    if os.path.isdir(c): NFI_DIR = c; break

if NFI_DIR is None:
    print('sign_data_nfi not found — skipping cross-dataset test.')
else:
    def _parse(fn):
        s = re.sub(r'^NFI-', '', os.path.basename(fn)); s = os.path.splitext(s)[0]
        return (s[:3], s[5:8]) if re.fullmatch(r'\d{8}', s) else None
    ng, nf = defaultdict(list), defaultdict(list)
    for f in glob.glob(os.path.join(NFI_DIR, 'genuine', '*')):
        p = _parse(f); ng[p[1]].append(f) if p else None
    for f in glob.glob(os.path.join(NFI_DIR, 'forged', '*')):
        p = _parse(f); nf[p[1]].append(f) if p else None
    owners = sorted(set(ng) & set(nf), key=int)

    # pair-based (like 3b) — uses global threshold
    rng = random.Random(7); nrows = []
    for o in owners:
        g = ng[o]
        if len(g) < 2: continue
        for _ in range(40): a, b = rng.sample(g, 2); nrows.append((a, b, 0))
        for _ in range(20):
            if nf.get(o): nrows.append((rng.choice(g), rng.choice(nf[o]), 1))
        for _ in range(20):
            other = rng.choice([x for x in owners if x != o]); nrows.append((rng.choice(g), rng.choice(ng[other]), 1))
    rng.shuffle(nrows)
    nfi_df = pd.DataFrame(nrows, columns=['img1','img2','label']); ny = nfi_df['label'].to_numpy()
    nd = pair_distances(nfi_df, root='')
    print('--- NFI pair-based (global threshold) ---')
    print('AUC', round(roc_auc_score(ny, nd), 3), '| EER', round(eer_threshold(nd, ny)[1]*100, 2), '%',
          '| at global thr -> FAR', round(100*(nd[ny==1] < threshold).mean(), 2),
          '% FRR', round(100*(nd[ny==0] > threshold).mean(), 2), '%')

    # per-writer adaptive on NFI (alpha from combined-val), scoring vs NFI enrollment refs
    rng = random.Random(13); sc, lb = [], []; ag = rf = ngq = nfq = 0
    for o in owners:
        g = ng[o]
        if len(g) < 6: continue
        refs = rng.sample(g, 5)
        re_emb = tower.predict(np.stack([to3(cv2.resize(cv2.imread(p, 0), (IMG_W, IMG_H))) for p in refs]), verbose=0)
        rd = np.sqrt(np.maximum(np.sum((re_emb[:,None]-re_emb[None,:])**2, -1), 0))
        iu = np.triu_indices(5, 1); intra = rd[iu]; tau = intra.mean() + alpha*intra.std()
        def sc1(p):
            e = tower.predict(to3(cv2.resize(cv2.imread(p, 0), (IMG_W, IMG_H)))[None], verbose=0)[0]
            return float(np.sqrt(np.sum((e - re_emb)**2, axis=1) + 1e-12).mean())
        for p in [x for x in g if x not in refs]: s = sc1(p); sc.append(s); lb.append(0); ngq+=1; ag += (s<tau)
        for p in nf.get(o, []): s = sc1(p); sc.append(s); lb.append(1); nfq+=1; rf += (s>=tau)
    sc = np.array(sc); lb = np.array(lb)
    print('--- NFI per-writer adaptive (alpha from val) ---')
    print('AUC', round(roc_auc_score(lb, sc), 3),
          '| FAR', round(100*(1-rf/max(nfq,1)), 2), '% | FRR', round(100*(1-ag/max(ngq,1)), 2), '%',
          f'(n={len(lb)})')